In [ ]:
%sql
-- compact the fact tables and co-locate the columns used by Power BI filters
USE CATALOG energy;
OPTIMIZE d1_gold.fact_generation_hourly ZORDER BY (date_key, zone_code);
OPTIMIZE d1_gold.fact_carbon_hourly ZORDER BY (date_key, zone_code);
OPTIMIZE d1_gold.fact_flow_hourly ZORDER BY (date_key, to_zone_code);

In [ ]:
%sql
-- rows per table
SELECT 'bronze_generation' AS table_name, COUNT(*) AS n_rows FROM d1_bronze.bronze_generation
UNION ALL SELECT 'bronze_load', COUNT(*) FROM d1_bronze.bronze_load
UNION ALL SELECT 'bronze_flows', COUNT(*) FROM d1_bronze.bronze_flows
UNION ALL SELECT 'silver_generation_hourly', COUNT(*) FROM d1_silver.silver_generation_hourly
UNION ALL SELECT 'silver_load_hourly', COUNT(*) FROM d1_silver.silver_load_hourly
UNION ALL SELECT 'silver_flows_hourly', COUNT(*) FROM d1_silver.silver_flows_hourly
UNION ALL SELECT 'fact_generation_hourly', COUNT(*) FROM d1_gold.fact_generation_hourly
UNION ALL SELECT 'fact_carbon_hourly', COUNT(*) FROM d1_gold.fact_carbon_hourly
UNION ALL SELECT 'fact_flow_hourly', COUNT(*) FROM d1_gold.fact_flow_hourly;

In [ ]:
%sql
-- duplicate keys (should be empty)
SELECT 'fact_carbon_hourly' AS table_name, zone_code AS key_1, CAST(ts_utc AS STRING) AS key_2, COUNT(*) AS n
FROM d1_gold.fact_carbon_hourly GROUP BY zone_code, ts_utc HAVING COUNT(*) > 1
UNION ALL
SELECT 'fact_flow_hourly', concat(from_zone_code, '>', to_zone_code), CAST(ts_utc AS STRING), COUNT(*)
FROM d1_gold.fact_flow_hourly GROUP BY from_zone_code, to_zone_code, ts_utc HAVING COUNT(*) > 1;

In [ ]:
%sql
-- coverage: traced hours, missing load, imports priced with a fallback intensity
SELECT
  zone_code,
  COUNT(*) AS n_hours,
  ROUND(AVG(CAST(is_flow_traced AS INT)), 3) AS share_traced,
  SUM(CASE WHEN load_mwh IS NULL THEN 1 ELSE 0 END) AS hours_without_load,
  ROUND(SUM(imports_fallback_ci_mwh) / NULLIF(SUM(imports_mwh), 0), 3) AS share_fallback_imports
FROM d1_gold.fact_carbon_hourly
GROUP BY zone_code
ORDER BY zone_code;

In [ ]:
%sql
-- orders of magnitude (life-cycle): FR ~ 30-80, BE ~ 100-200, NL ~ 250-400, DE ~ 300-450 g/kWh
SELECT
  zone_code,
  ROUND(SUM(emissions_production_lca_t) * 1000 / SUM(production_mwh)) AS ci_production_lca,
  ROUND(SUM(emissions_consumption_lca_t) * 1000 / SUM(load_mwh)) AS ci_consumption_lca,
  ROUND(SUM(renewable_mwh) / SUM(production_mwh), 3) AS renewable_share,
  ROUND(SUM(net_imports_mwh) / SUM(load_mwh), 3) AS net_import_rate
FROM d1_gold.fact_carbon_hourly
WHERE is_flow_traced
GROUP BY zone_code
ORDER BY zone_code;